# Exploratory platform interaction analysis across metrics

This notebook estimates whether treatment effects differ between Mobile and Desktop for five ratio metrics.

For every metric, the target quantity is:

(Metric_B - Metric_A)_Mobile - (Metric_B - Metric_A)_Desktop

This is an exploratory interaction analysis. It uses all available platform blocks and does not restrict the primary analysis to a fully matched Mobile/Desktop subset.

In [1]:
import json

import numpy as np
import pandas as pd

DATA_PATH = 'csv/marketpele_ab_test_data_set.csv'
BLOCK_COLS = ['date', 'publisher_id', 'platform']
CLUSTER_COLS = ['date', 'publisher_id']
GROUPS = ['A', 'B']
PLATFORMS = ['Mobile', 'Desktop']

METRIC_CONFIGS = [
    ('RPM', 'revenue', 'pageviews', 1000),
    ('RPS', 'revenue', 'sessions', 1000),
    ('Paid CTR', 'sponsord_clicks', 'visible_pageviews', 100),
    ('Organic CTR', 'organic_clicks', 'visible_pageviews', 100),
    ('Pages per Session', 'pageviews', 'sessions', 1),
]
METRIC_CONFIG_BY_NAME = {
    metric_name: {
        'numerator': numerator,
        'denominator': denominator,
        'multiplier': multiplier,
    }
    for metric_name, numerator, denominator, multiplier in METRIC_CONFIGS
}
VALUE_COLUMNS = sorted({
    column
    for _, numerator, denominator, _ in METRIC_CONFIGS
    for column in (numerator, denominator)
})

df = pd.read_csv(DATA_PATH, sep=',')
df.columns = df.columns.str.strip()

for column in VALUE_COLUMNS:
    if pd.api.types.is_string_dtype(df[column]):
        df[column] = (
            df[column]
            .str.replace('\u00a0', '', regex=False)
            .str.replace(r'\s+', '', regex=True)
        )
    if column == 'revenue':
        df[column] = df[column].str.replace('$', '', regex=False).str.replace(',', '.', regex=False)
    df[column] = pd.to_numeric(df[column], errors='raise')

duplicate_blocks = df.duplicated(
    subset=BLOCK_COLS + ['group_name']
).sum()
assert duplicate_blocks == 0, (
    'The input is expected to contain one row per '
    'date × publisher_id × platform × group_name block.'
)
assert set(df['group_name']) == set(GROUPS)
assert set(df['platform']) == set(PLATFORMS)
assert df[VALUE_COLUMNS].notna().all().all()

print(f'Input rows: {len(df)}')
print(f'Available blocks: {df[BLOCK_COLS].drop_duplicates().shape[0]}')
print(df.groupby(['platform', 'group_name']).size().rename('rows').to_string())
print('Metric configurations:')
print(pd.DataFrame(METRIC_CONFIGS, columns=['metric', 'numerator', 'denominator', 'multiplier']).to_string(index=False))

Input rows: 126
Available blocks: 63
platform  group_name
Desktop   A             35
          B             35
Mobile    A             28
          B             28
Metric configurations:
           metric       numerator       denominator  multiplier
              RPM         revenue         pageviews        1000
              RPS         revenue          sessions        1000
         Paid CTR sponsord_clicks visible_pageviews         100
      Organic CTR  organic_clicks visible_pageviews         100
Pages per Session       pageviews          sessions           1


## Bootstrap design

The source data are already aggregated at `date × publisher_id × platform × group_name`, so the analysis does not re-aggregate them. A pivot below only aligns the existing A and B rows within each block.

The resampling unit is the `date × publisher_id` cluster. One common bootstrap sample draws the available clusters with replacement, in the same number as the original clusters. The sampled cluster multiplicity is applied to every available platform block inside that cluster: when both Mobile and Desktop are present, both are kept together; when only one platform is present, only that platform contributes. Within every `date × publisher_id × platform` block, A and B remain paired.

This cluster-level resampling preserves the observed cross-platform dependence for clusters containing both platforms while using all available data. It also preserves the incomplete platform composition: clusters with only one platform remain one-platform clusters in the bootstrap samples.

For every bootstrap replicate and every metric, the metric is calculated as a ratio of sums using the configuration below. The interaction is calculated directly as the Mobile treatment effect minus the Desktop treatment effect. The reported interval is the 95% percentile bootstrap confidence interval for that absolute-scale interaction.

Metric formulas reused from the existing analysis notebooks:

- RPM: `revenue / pageviews × 1000`
- RPS: `revenue / sessions × 1000`
- Paid CTR: `sponsord_clicks / visible_pageviews × 100`
- Organic CTR: `organic_clicks / visible_pageviews × 100`
- Pages per Session: `pageviews / sessions × 1`

Candidate bootstrap draws with a zero or non-finite denominator for any metric/platform are discarded and replaced by new common cluster draws. The code raises an error if an invalid denominator reaches the ratio calculation; it never silently returns infinite values.

In [2]:
# Align the existing A/B rows within each available platform block.
# This pivot only aligns already-aggregated rows; it does not re-aggregate input data.
aligned_blocks = (
    df[BLOCK_COLS + ['group_name'] + VALUE_COLUMNS]
    .pivot(index=BLOCK_COLS, columns='group_name', values=VALUE_COLUMNS)
    .reset_index()
)

aligned_blocks.columns = [
    '_'.join(str(part) for part in column if str(part) != '')
    for column in aligned_blocks.columns.to_flat_index()
]

required_columns = (
    BLOCK_COLS
    + [f'{column}_{group}' for column in VALUE_COLUMNS for group in GROUPS]
)
missing_columns = sorted(set(required_columns) - set(aligned_blocks.columns))
assert not missing_columns, f'Missing A/B columns after pivot: {missing_columns}'
assert not aligned_blocks[required_columns].isna().any().any(), (
    'Every available platform block must contain both A and B for all metrics.'
)

aligned_blocks = aligned_blocks.sort_values(BLOCK_COLS).reset_index(drop=True)

print('Blocks by platform:')
print(aligned_blocks.groupby('platform').size().to_string())

# Reshape existing platform blocks into date × publisher_id clusters.
# Platform blocks are kept together within shared clusters during bootstrap.
cluster_value_columns = [
    f'{column}_{group}'
    for column in VALUE_COLUMNS
    for group in GROUPS
]

metric_clusters = (
    aligned_blocks[CLUSTER_COLS]
    .drop_duplicates()
    .sort_values(CLUSTER_COLS)
    .reset_index(drop=True)
)

for platform in PLATFORMS:
    platform_values = aligned_blocks.loc[
        aligned_blocks['platform'] == platform,
        CLUSTER_COLS + cluster_value_columns,
    ].rename(
        columns={column: f'{column}_{platform}' for column in cluster_value_columns}
    )
    metric_clusters = metric_clusters.merge(
        platform_values,
        on=CLUSTER_COLS,
        how='left',
        validate='one_to_one',
    )

platform_presence = pd.DataFrame({
    platform: metric_clusters[f'pageviews_A_{platform}'].notna()
    for platform in PLATFORMS
})
for platform in PLATFORMS:
    platform_columns = [f'{column}_{platform}' for column in cluster_value_columns]
    present = platform_presence[platform]
    assert metric_clusters.loc[present, platform_columns].notna().all().all()
    metric_clusters[platform_columns] = metric_clusters[platform_columns].fillna(0.0)

print(f'date × publisher_id clusters: {len(metric_clusters)}')
print('Number of available platforms per cluster:')
print(platform_presence.sum(axis=1).value_counts().sort_index().to_string())
metric_clusters.head()

Blocks by platform:
platform
Desktop    35
Mobile     28
date × publisher_id clusters: 56
Number of available platforms per cluster:
1    49
2     7


,date,publisher_id,organic_clicks_A_Mobile,organic_clicks_B_Mobile,pageviews_A_Mobile,pageviews_B_Mobile,revenue_A_Mobile,revenue_B_Mobile,sessions_A_Mobile,sessions_B_Mobile,...,pageviews_A_Desktop,pageviews_B_Desktop,revenue_A_Desktop,revenue_B_Desktop,sessions_A_Desktop,sessions_B_Desktop,sponsord_clicks_A_Desktop,sponsord_clicks_B_Desktop,visible_pageviews_A_Desktop,visible_pageviews_B_Desktop
0,2019-03-01,101,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,...,16580.0,16191.0,34.52,32.92,12684.0,12717.0,300.0,268.0,5418.0,4906.0
1,2019-03-01,106,1600.0,672.0,30494.0,30079.0,82.93,100.82,22843.0,23103.0,...,16227.0,15060.0,20.16,19.74,11750.0,11967.0,601.0,663.0,11395.0,10369.0
2,2019-03-01,111,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,...,12876.0,12135.0,68.39,59.55,8664.0,8524.0,322.0,278.0,5028.0,4458.0
3,2019-03-01,123,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,...,26095.0,25546.0,11.52,11.59,19920.0,19915.0,298.0,320.0,4409.0,4118.0
4,2019-03-01,259,2536.0,1171.0,59977.0,58395.0,10.95,11.35,20649.0,20603.0,...,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
def _ratio_from_cluster_indices(
    clusters,
    indices,
    platform,
    numerator,
    denominator,
    group,
    multiplier,
):
    """Calculate a ratio metric as a ratio of sums for sampled clusters."""
    numerator_values = clusters[f'{numerator}_{group}_{platform}'].to_numpy(dtype=float)[indices]
    denominator_values = clusters[f'{denominator}_{group}_{platform}'].to_numpy(dtype=float)[indices]
    denominator_sums = denominator_values.sum(axis=1)

    if np.any(~np.isfinite(denominator_sums)) or np.any(denominator_sums <= 0):
        raise ValueError(
            f'Non-positive or non-finite bootstrap denominator for {platform}: '
            f'{denominator}'
        )

    metric_values = numerator_values.sum(axis=1) / denominator_sums * multiplier
    if not np.isfinite(metric_values).all():
        raise ValueError(f'Non-finite bootstrap metric for {platform}: {numerator}/{denominator}')
    return metric_values


def _observed_ratio_metric(clusters, platform, numerator, denominator, group, multiplier):
    numerator_sum = clusters[f'{numerator}_{group}_{platform}'].sum()
    denominator_sum = clusters[f'{denominator}_{group}_{platform}'].sum()
    if not np.isfinite(denominator_sum) or denominator_sum <= 0:
        raise ValueError(f'Non-positive or non-finite observed denominator for {platform}: {denominator}')
    metric_value = numerator_sum / denominator_sum * multiplier
    if not np.isfinite(metric_value):
        raise ValueError(f'Non-finite observed metric for {platform}: {numerator}/{denominator}')
    return float(metric_value)


def platform_interaction_bootstrap(
    blocks,
    clusters,
    metric_configs,
    n_bootstrap=10_000,
    alpha=0.05,
    seed=42,
):
    """Estimate direct Mobile-minus-Desktop interactions for ratio metrics."""
    if n_bootstrap <= 0:
        raise ValueError('n_bootstrap must be positive')
    if not 0 < alpha < 0.5:
        raise ValueError('alpha must be between 0 and 0.5')

    rng = np.random.default_rng(seed)
    n_clusters = len(clusters)
    if n_clusters == 0:
        raise ValueError('No date × publisher_id clusters found')

    observed_results = {}
    for metric_name, numerator, denominator, multiplier in metric_configs:
        observed_results[metric_name] = {}
        for platform in PLATFORMS:
            platform_blocks = blocks[blocks['platform'] == platform]
            if platform_blocks.empty:
                raise ValueError(f'No blocks found for {platform}')
            metric_a = _observed_ratio_metric(
                clusters, platform, numerator, denominator, 'A', multiplier
            )
            metric_b = _observed_ratio_metric(
                clusters, platform, numerator, denominator, 'B', multiplier
            )
            observed_results[metric_name][platform] = {
                'metric_a': metric_a,
                'metric_b': metric_b,
                'treatment_effect': metric_b - metric_a,
                'n_blocks': int(len(platform_blocks)),
            }

    # A common cluster draw is reused for all platforms and all metrics.
    accepted_draws = 0
    n_rejected_bootstrap_draws = 0
    bootstrap_effect_arrays = {metric_name: [] for metric_name, *_ in metric_configs}
    while accepted_draws < n_bootstrap:
        batch_size = max(1024, n_bootstrap - accepted_draws)
        sample_indices = rng.integers(
            0,
            n_clusters,
            size=(batch_size, n_clusters),
        )
        valid_draw = np.ones(batch_size, dtype=bool)

        for metric_name, numerator, denominator, multiplier in metric_configs:
            for platform in PLATFORMS:
                denominator_values = clusters[
                    f'{denominator}_A_{platform}'
                ].to_numpy(dtype=float)[sample_indices]
                denominator_sums = denominator_values.sum(axis=1)
                valid_draw &= np.isfinite(denominator_sums) & (denominator_sums > 0)

                denominator_values = clusters[
                    f'{denominator}_B_{platform}'
                ].to_numpy(dtype=float)[sample_indices]
                denominator_sums = denominator_values.sum(axis=1)
                valid_draw &= np.isfinite(denominator_sums) & (denominator_sums > 0)

        n_rejected_bootstrap_draws += int((~valid_draw).sum())
        valid_indices = sample_indices[valid_draw]
        if len(valid_indices) == 0:
            continue

        for metric_name, numerator, denominator, multiplier in metric_configs:
            metric_effects = {}
            for platform in PLATFORMS:
                bootstrap_a = _ratio_from_cluster_indices(
                    clusters, valid_indices, platform, numerator, denominator, 'A', multiplier
                )
                bootstrap_b = _ratio_from_cluster_indices(
                    clusters, valid_indices, platform, numerator, denominator, 'B', multiplier
                )
                metric_effects[platform] = bootstrap_b - bootstrap_a
            bootstrap_effect_arrays[metric_name].append(
                metric_effects['Mobile'] - metric_effects['Desktop']
            )

        accepted_draws += len(valid_indices)
    print(f'Rejected bootstrap draws: {n_rejected_bootstrap_draws}')

    results = {}
    for metric_name, *_ in metric_configs:
        interaction_bootstrap = np.concatenate(bootstrap_effect_arrays[metric_name])[:n_bootstrap]
        ci_lower, ci_upper = np.quantile(
            interaction_bootstrap,
            [alpha / 2, 1 - alpha / 2],
        )
        mobile = observed_results[metric_name]['Mobile']
        desktop = observed_results[metric_name]['Desktop']
        interaction = mobile['treatment_effect'] - desktop['treatment_effect']
        results[metric_name] = {
            'metric': metric_name,
            'Mobile': mobile,
            'Desktop': desktop,
            'interaction': float(interaction),
            'interaction_ci_lower': float(ci_lower),
            'interaction_ci_upper': float(ci_upper),
            'ci_crosses_zero': bool(ci_lower <= 0 <= ci_upper),
            'n_mobile_blocks': mobile['n_blocks'],
            'n_desktop_blocks': desktop['n_blocks'],
            'n_bootstrap': n_bootstrap,
            'n_clusters': n_clusters,
            'n_rejected_bootstrap_draws': n_rejected_bootstrap_draws,
            'bootstrap_interaction': interaction_bootstrap,
        }

    return results

In [4]:
interaction_results = platform_interaction_bootstrap(
    blocks=aligned_blocks,
    clusters=metric_clusters,
    metric_configs=METRIC_CONFIGS,
    n_bootstrap=10_000,
    alpha=0.05,
    seed=42,
)

summary_rows = []
for metric_name, *_ in METRIC_CONFIGS:
    result = interaction_results[metric_name]
    summary_rows.append({
        'Metric': metric_name,
        'Mobile A': result['Mobile']['metric_a'],
        'Mobile B': result['Mobile']['metric_b'],
        'Mobile effect': result['Mobile']['treatment_effect'],
        'Desktop A': result['Desktop']['metric_a'],
        'Desktop B': result['Desktop']['metric_b'],
        'Desktop effect': result['Desktop']['treatment_effect'],
        'Interaction': result['interaction'],
        'CI lower': result['interaction_ci_lower'],
        'CI upper': result['interaction_ci_upper'],
        'CI crosses zero': result['ci_crosses_zero'],
    })

interaction_summary = pd.DataFrame(summary_rows)
display(interaction_summary)

json_results = {}
for metric_name, result in interaction_results.items():
    json_results[metric_name] = {
        'metric': result['metric'],
        'mobile_a': result['Mobile']['metric_a'],
        'mobile_b': result['Mobile']['metric_b'],
        'mobile_treatment_effect': result['Mobile']['treatment_effect'],
        'desktop_a': result['Desktop']['metric_a'],
        'desktop_b': result['Desktop']['metric_b'],
        'desktop_treatment_effect': result['Desktop']['treatment_effect'],
        'observed_interaction': result['interaction'],
        'ci_lower': result['interaction_ci_lower'],
        'ci_upper': result['interaction_ci_upper'],
        'ci_crosses_zero': result['ci_crosses_zero'],
        'n_mobile_blocks': result['n_mobile_blocks'],
        'n_desktop_blocks': result['n_desktop_blocks'],
        'n_bootstrap': result['n_bootstrap'],
        'n_clusters': result['n_clusters'],
        'n_rejected_bootstrap_draws': result['n_rejected_bootstrap_draws'],
    }

with open('outputs/platform_interaction_results.json', 'w', encoding='utf-8') as output_file:
    json.dump(json_results, output_file, indent=2)

Rejected bootstrap draws: 0


,Metric,Mobile A,Mobile B,Mobile effect,Desktop A,Desktop B,Desktop effect,Interaction,CI lower,CI upper,CI crosses zero
0,RPM,1.485335,1.831176,0.345842,1.688332,1.766317,0.077985,0.267856,-0.006654,0.594956,True
1,RPS,2.553007,3.032735,0.479728,3.085161,3.049368,-0.035793,0.515521,0.064449,1.064653,False
2,Paid CTR,6.651732,8.437994,1.786262,5.347176,6.183460,0.836284,0.949978,0.391943,1.570758,False
3,Organic CTR,22.024666,11.703241,-10.321425,19.189740,9.880710,-9.309031,-1.012394,-1.977339,-0.046490,False
4,Pages per Session,1.718809,1.656168,-0.062642,1.827343,1.726399,-0.100944,0.038302,-0.034700,0.149959,True


## Interpretation

Each row reports an absolute-scale interaction:

`(Metric_B - Metric_A)_Mobile - (Metric_B - Metric_A)_Desktop`.

The interaction is evaluated directly through its bootstrap confidence interval; separate platform significance results are not used as evidence that platform effects differ.

The direct interaction analysis provides evidence of platform heterogeneity for **RPS, Paid CTR, and Organic CTR**. RPS and Paid CTR show stronger positive treatment effects on Mobile than on Desktop, while Organic CTR shows a stronger negative treatment effect on Mobile.

For **RPM** and **Pages per Session**, the bootstrap confidence intervals include zero, so the available data do not provide sufficient evidence that the treatment effects differ between Mobile and Desktop for these metrics.

For Paid CTR and Organic CTR, the multiplier is 100, so absolute treatment effects and interactions are expressed in **percentage points**, not relative percentages.

The full bootstrap interaction distributions remain available in `interaction_results[metric]['bootstrap_interaction']`, while `interaction_summary` provides the compact human-readable results table.

Mobile and Desktop are not fully matched by `date × publisher_id`, so the observed interactions may partly reflect differences in publisher/date composition between the platform samples. Therefore, these results provide exploratory evidence of platform heterogeneity for some metrics, but should not be interpreted as clean causal estimates of platform moderation.